In [2]:
import ee
import datetime
import pandas as pd
import requests
import rasterio
import os
from math import radians, sin, cos, atan2, sqrt

In [9]:

annual_emissions_df = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-2019-2024.csv')
facility_attributes_df = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/facility-attributes-2019-2024.csv')
location_cols = ['State', 'Facility Name', 'Facility ID', 'Unit ID', 'Year', 
                 'Latitude', 'Longitude', 'County', 'EPA Region', 'NERC Region']

merged_df = annual_emissions_df.merge(
    facility_attributes_df[location_cols],
    on=['State', 'Facility Name', 'Facility ID', 'Unit ID', 'Year'],
    how='left'
)
merged_df.to_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/unit-emissions-with-locations-2019-2024.csv', index=False)

In [28]:
import pandas as pd
import numpy as np

# Read the data
annual_emissions_df = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/unit-emissions-with-locations-2019-2024.csv')

def get_facility_primary_fuel(group):
    """
    Determine primary fuel type for a facility-year combination.
    Uses heat input-weighted approach - fuel type with most heat input is primary.
    """
    # Remove rows with missing heat input
    valid_data = group.dropna(subset=['Heat Input (mmBtu)'])
    
    if len(valid_data) == 0:
        return 'Unknown'
    
    # Sum heat input by fuel type
    fuel_heat = valid_data.groupby('Primary Fuel Type')['Heat Input (mmBtu)'].sum()
    
    # Return fuel type with maximum heat input
    return fuel_heat.idxmax()

# Aggregate to facility-year level (group by Facility ID and Year only)
facility_aggregated = annual_emissions_df.groupby(['Facility ID', 'Year']).agg({
    # Operating metrics
    'Operating Time Count': 'sum',
    'Sum of the Operating Time': 'sum',
    
    # Generation and load
    'Gross Load (MWh)': 'sum',
    'Steam Load (1000 lb)': 'sum',
    
    # Emissions
    'SO2 Mass (short tons)': 'sum',
    'CO2 Mass (short tons)': 'sum',
    'NOx Mass (short tons)': 'sum',
    
    # Heat input
    'Heat Input (mmBtu)': 'sum',
    
    # Count units
    'Unit ID': 'count'  # This gives us number of units
}).reset_index()

# Rename unit count column
facility_aggregated.rename(columns={'Unit ID': 'Unit Count'}, inplace=True)

# Get facility metadata separately (use first occurrence)
facility_info = annual_emissions_df.groupby(['Facility ID', 'Year']).agg({
    'Facility Name': 'first',
    'State': 'first', 
    'Latitude': 'first',
    'Longitude': 'first',
    'County': 'first',
    'EPA Region': 'first',
    'NERC Region': 'first'
}).reset_index()

# Merge metadata back
facility_aggregated = facility_aggregated.merge(facility_info, on=['Facility ID', 'Year'])

# Get primary fuel type for each facility-year
fuel_by_facility_year = annual_emissions_df.groupby(['Facility ID', 'Year']).apply(get_facility_primary_fuel).reset_index()
fuel_by_facility_year.columns = ['Facility ID', 'Year', 'Primary Fuel Type']

# Merge primary fuel type back
facility_aggregated = facility_aggregated.merge(fuel_by_facility_year, on=['Facility ID', 'Year'], how='left')

# Calculate weighted average emission rates
facility_aggregated['SO2 Rate (lbs/mmBtu)'] = np.where(
    facility_aggregated['Heat Input (mmBtu)'] > 0,
    (facility_aggregated['SO2 Mass (short tons)'] * 2000) / facility_aggregated['Heat Input (mmBtu)'],
    np.nan
)

facility_aggregated['CO2 Rate (short tons/mmBtu)'] = np.where(
    facility_aggregated['Heat Input (mmBtu)'] > 0,
    facility_aggregated['CO2 Mass (short tons)'] / facility_aggregated['Heat Input (mmBtu)'],
    np.nan
)

facility_aggregated['NOx Rate (lbs/mmBtu)'] = np.where(
    facility_aggregated['Heat Input (mmBtu)'] > 0,
    (facility_aggregated['NOx Mass (short tons)'] * 2000) / facility_aggregated['Heat Input (mmBtu)'],
    np.nan
)

# Sort by facility and year
facility_aggregated.sort_values(['Facility ID', 'Year'], inplace=True)

# Display summary
print(f"Original unit-level records: {len(annual_emissions_df)}")
print(f"Aggregated facility-year records: {len(facility_aggregated)}")
print(f"\nFacilities with multiple fuel types:")

# Check facilities that have different primary fuels across years
fuel_changes = facility_aggregated.groupby('Facility ID')['Primary Fuel Type'].nunique()
multi_fuel_facilities = fuel_changes[fuel_changes > 1].index
for fac_id in list(multi_fuel_facilities)[:5]:  # Show first 5 examples
    fac_data = facility_aggregated[facility_aggregated['Facility ID'] == fac_id]
    print(f"\nFacility {fac_data['Facility Name'].iloc[0]} (ID: {fac_id}):")
    print(fac_data[['Year', 'Primary Fuel Type', 'Heat Input (mmBtu)']].to_string(index=False))

# Save aggregated data
facility_aggregated.to_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/facility-emissions-aggregated-2019-2024.csv', index=False)

# Create a summary by fuel type and year
fuel_summary = facility_aggregated.groupby(['Year', 'Primary Fuel Type']).agg({
    'Facility ID': 'count',
    'Gross Load (MWh)': 'sum',
    'CO2 Mass (short tons)': 'sum',
    'SO2 Mass (short tons)': 'sum',
    'NOx Mass (short tons)': 'sum'
}).round(2)

print("\n\nEmissions Summary by Fuel Type and Year:")
print(fuel_summary)

print(f"\nSuccessfully saved {len(facility_aggregated)} facility-year records to file")

Original unit-level records: 24996
Aggregated facility-year records: 8250

Facilities with multiple fuel types:

Facility Gadsden (ID: 7):
 Year    Primary Fuel Type  Heat Input (mmBtu)
 2019 Pipeline Natural Gas          2962336.65
 2020 Pipeline Natural Gas           886757.55
 2021              Unknown                0.00
 2022 Pipeline Natural Gas            20971.00

Facility Charles R Lowman (ID: 56):
 Year    Primary Fuel Type  Heat Input (mmBtu)
 2019                 Coal        10455835.067
 2020                 Coal         4043808.636
 2023 Pipeline Natural Gas        10871305.052
 2024 Pipeline Natural Gas        26781923.173

Facility Seminole (136) (ID: 136):
 Year Primary Fuel Type  Heat Input (mmBtu)
 2019              Coal        72081531.054
 2020              Coal        68315035.552
 2021              Coal        65596536.064
 2022              Coal        61933564.347
 2023              Coal        79255280.689
 2024       Natural Gas        77785902.600

Facility 

/tmp/ipykernel_917614/1390970198.py:64: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fuel_by_facility_year = annual_emissions_df.groupby(['Facility ID', 'Year']).apply(get_facility_primary_fuel).reset_index()


In [27]:
facility_aggregated

,Facility ID,Year,Operating Time Count,Sum of the Operating Time,Gross Load (MWh),Steam Load (1000 lb),SO2 Mass (short tons),CO2 Mass (short tons),NOx Mass (short tons),Heat Input (mmBtu),...,State,Latitude,Longitude,County,EPA Region,NERC Region,Primary Fuel Type,SO2 Rate (lbs/mmBtu),CO2 Rate (short tons/mmBtu),NOx Rate (lbs/mmBtu)
0,3,2019,44642,44567.00,12971220.25,0.00,3494.672,8290059.850,2361.074,1.060760e+08,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.065890,0.078152,0.044517
1,3,2020,38901,38848.75,11671959.25,0.00,1283.482,6846726.200,1656.340,9.189587e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.027933,0.074505,0.036048
2,3,2021,41030,40957.25,12221921.00,0.00,3881.670,7577398.025,2365.775,9.791375e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.079288,0.077388,0.048324
3,3,2022,43760,43632.50,12334645.50,0.00,1349.396,7588387.100,1941.880,9.964551e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.027084,0.076154,0.038976
4,3,2023,37933,37866.00,9909175.25,0.00,280.447,5492459.550,1325.482,7.803848e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.007187,0.070381,0.033970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8245,880109,2023,3672,3672.00,0.00,0.00,0.000,0.000,11.016,1.220206e+06,...,OH,40.5426,-84.1916,Auglaize County,5,None,Pipeline Natural Gas,0.000000,0.000000,0.018056
8246,880109,2024,3672,3672.00,0.00,0.00,0.000,0.000,11.016,1.220206e+06,...,OH,40.5426,-84.1916,Auglaize County,5,None,Pipeline Natural Gas,0.000000,0.000000,0.018056
8247,880110,2022,5824,5792.42,0.00,321550.29,0.000,0.000,644.406,1.938144e+06,...,TN,36.5493,-82.6342,Hawkins County,4,None,Pipeline Natural Gas,0.000000,0.000000,0.664972
8248,880110,2023,7434,7387.48,0.00,434323.10,0.000,0.000,126.504,6.242826e+05,...,TN,36.5493,-82.6342,Hawkins County,4,None,Pipeline Natural Gas,0.000000,0.000000,0.405278


In [14]:
import pandas as pd
import numpy as np

def analyze_facilities_by_total_emissions(csv_path):
    """
    Analyze each power plant's total emissions across all years.
    
    Args:
        csv_path (str): Path to the CSV file containing facility emissions data
    
    Returns:
        pd.DataFrame: Aggregated results by facility with rankings and statistics
    """
    
    # Read the CSV file
    print(f"Reading data from: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # Display basic info about the dataset
    print(f"Dataset shape: {df.shape}")
    print(f"Years covered: {df['Year'].min()} - {df['Year'].max()}")
    print(f"Number of unique facilities: {df['Facility ID'].nunique()}")
    print(f"Total facility-year records: {len(df)}")
    
    # Helper function for safe weighted average
    def safe_weighted_average(values, weights):
        """Calculate weighted average, handling zero weights."""
        # Filter out zero weights and corresponding values
        valid_mask = weights > 0
        if valid_mask.sum() == 0:  # All weights are zero
            return np.mean(values) if len(values) > 0 else 0
        return np.average(values[valid_mask], weights=weights[valid_mask])
    
    # Group by facility and aggregate all metrics across years
    # Include Latitude and Longitude, taking the mean for each facility
    facility_totals = df.groupby(['Facility Name', 'Facility ID', 'State']).agg({
        # Geographic coordinates - take mean in case of slight variations
        'Latitude': 'mean',
        'Longitude': 'mean',
        
        # Direct sums for mass/energy metrics across all years
        'Operating Time Count': 'sum',
        'Sum of the Operating Time': 'sum', 
        'Gross Load (MWh)': 'sum',
        'Steam Load (1000 lb)': 'sum',
        'SO2 Mass (short tons)': 'sum',
        'CO2 Mass (short tons)': 'sum',
        'NOx Mass (short tons)': 'sum',
        'Heat Input (mmBtu)': 'sum',
        'Unit Count': 'mean',  # Average units per year
        
        # For rates, calculate weighted averages using Heat Input as weight
        'SO2 Rate (lbs/mmBtu)': lambda x: safe_weighted_average(
            x.values, df.loc[x.index, 'Heat Input (mmBtu)'].values),
        'CO2 Rate (short tons/mmBtu)': lambda x: safe_weighted_average(
            x.values, df.loc[x.index, 'Heat Input (mmBtu)'].values),
        'NOx Rate (lbs/mmBtu)': lambda x: safe_weighted_average(
            x.values, df.loc[x.index, 'Heat Input (mmBtu)'].values),
        
        # Operational info
        'Year': ['min', 'max', 'count'],  # First year, last year, years of operation
        'Primary Fuel Type': lambda x: x.mode().iloc[0] if not x.mode().empty else 'Mixed'
    }).reset_index()
    
    # Flatten multi-level columns from Year aggregation
    facility_totals.columns = [
        'Facility_Name', 'Facility_ID', 'State', 'Latitude', 'Longitude',
        'Total_Operating_Time_Count', 'Total_Operating_Time_Sum', 'Total_Gross_Load_MWh',
        'Total_Steam_Load_1000lb', 'Total_SO2_Mass', 'Total_CO2_Mass', 'Total_NOx_Mass',
        'Total_Heat_Input_mmBtu', 'Avg_Unit_Count', 'Avg_SO2_Rate', 'Avg_CO2_Rate',
        'Avg_NOx_Rate', 'First_Year', 'Last_Year', 'Years_in_Operation', 'Primary_Fuel_Type'
    ]
    
    # Sort by NOx emissions (descending) to get rankings
    facility_totals = facility_totals.sort_values('Total_NOx_Mass', ascending=False).reset_index(drop=True)
    
    # Add NOx ranking
    facility_totals['NOx_Rank'] = range(1, len(facility_totals) + 1)
    
    # Calculate totals for percentage calculations
    mass_columns = ['Total_Operating_Time_Count', 'Total_Operating_Time_Sum', 'Total_Gross_Load_MWh', 
                   'Total_Steam_Load_1000lb', 'Total_SO2_Mass', 'Total_CO2_Mass', 'Total_NOx_Mass', 
                   'Total_Heat_Input_mmBtu']
    
    totals = {}
    for col in mass_columns:
        totals[col] = facility_totals[col].sum()
    
    # Calculate cumulative sums and percentages for all mass/energy metrics
    for col in mass_columns:
        # Cumulative sum based on NOx ranking
        facility_totals[f'Cumulative_{col}'] = facility_totals[col].cumsum()
        # Individual percentage of total
        facility_totals[f'{col}_Percentage'] = (facility_totals[col] / totals[col] * 100).round(3)
        # Cumulative percentage
        facility_totals[f'Cumulative_{col}_Percentage'] = (facility_totals[f'Cumulative_{col}'] / totals[col] * 100).round(3)
    
    # Add operational period
    facility_totals['Operation_Period'] = facility_totals['Last_Year'] - facility_totals['First_Year'] + 1
    
    # Round numerical columns for better display
    numeric_columns = facility_totals.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if 'Percentage' in col or 'Rate' in col:
            facility_totals[col] = facility_totals[col].round(3)
        elif col in ['Latitude', 'Longitude']:
            facility_totals[col] = facility_totals[col].round(6)  # Higher precision for coordinates
        else:
            facility_totals[col] = facility_totals[col].round(2)
    
    return facility_totals

def print_facility_summary(results_df):
    """Print comprehensive summary statistics by facility."""
    
    print("\n" + "="*160)
    print("POWER PLANT EMISSIONS ANALYSIS - RANKED BY TOTAL NOx EMISSIONS")
    print("="*160)
    
    # Top facilities table with coordinates
    core_columns = ['NOx_Rank', 'Facility_Name', 'State', 'Latitude', 'Longitude', 
                   'Total_NOx_Mass', 'Total_CO2_Mass', 'Total_SO2_Mass', 
                   'Total_Heat_Input_mmBtu', 'Primary_Fuel_Type', 
                   'Years_in_Operation', 'First_Year', 'Last_Year']
    
    print("\nTOP EMITTING FACILITIES:")
    print("-" * 160)
    top_facilities = results_df[core_columns].head(15)
    print(top_facilities.to_string(index=False))
    
    # Percentage contributions for top facilities
    percentage_columns = ['NOx_Rank', 'Facility_Name', 'Latitude', 'Longitude',
                         'Total_NOx_Mass_Percentage', 'Total_CO2_Mass_Percentage', 
                         'Total_SO2_Mass_Percentage', 'Total_Heat_Input_mmBtu_Percentage']
    
    print("\n\nPERCENTAGE CONTRIBUTIONS (Top 10):")
    print("-" * 120)
    percentage_table = results_df[percentage_columns].head(10)
    print(percentage_table.to_string(index=False))
    
    # Operating metrics for top facilities
    operating_columns = ['NOx_Rank', 'Facility_Name', 'Latitude', 'Longitude',
                        'Total_Operating_Time_Count', 'Total_Gross_Load_MWh', 
                        'Avg_Unit_Count', 'Operation_Period']
    
    print("\n\nOPERATING METRICS (Top 10):")
    print("-" * 120)
    operating_table = results_df[operating_columns].head(10)
    print(operating_table.to_string(index=False))
    
    # Emission rates for top facilities
    rate_columns = ['NOx_Rank', 'Facility_Name', 'Latitude', 'Longitude',
                   'Avg_SO2_Rate', 'Avg_CO2_Rate', 'Avg_NOx_Rate']
    
    print("\n\nEMISSION RATES (Top 10):")
    print("-" * 100)
    rate_table = results_df[rate_columns].head(10)
    print(rate_table.to_string(index=False))
    
    print("\n" + "="*160)
    print("KEY INSIGHTS")
    print("="*160)
    
    # Top emitters
    top_facility = results_df.iloc[0]
    print(f"HIGHEST NOx EMITTER:")
    print(f"• {top_facility['Facility_Name']}, {top_facility['State']}")
    print(f"• Location: {top_facility['Latitude']:.6f}°N, {top_facility['Longitude']:.6f}°W")
    print(f"• Total NOx: {top_facility['Total_NOx_Mass']:,.2f} tons ({top_facility['Total_NOx_Mass_Percentage']:.2f}%)")
    print(f"• Fuel: {top_facility['Primary_Fuel_Type']}")
    print(f"• Operating period: {top_facility['First_Year']}-{top_facility['Last_Year']} ({top_facility['Years_in_Operation']} years)")
    
    # Top 5 contribution
    top5_contribution = results_df.head(5)['Total_NOx_Mass_Percentage'].sum()
    print(f"\nTOP 5 FACILITIES:")
    print(f"• Account for {top5_contribution:.1f}% of total NOx emissions")
    print(f"• Names: {', '.join(results_df.head(5)['Facility_Name'].tolist())}")
    
    # Geographic distribution
    print(f"\nGEOGRAPHIC DISTRIBUTION:")
    print(f"• Latitude range: {results_df['Latitude'].min():.3f}° to {results_df['Latitude'].max():.3f}°")
    print(f"• Longitude range: {results_df['Longitude'].min():.3f}° to {results_df['Longitude'].max():.3f}°")
    
    # Fuel type analysis
    print(f"\nFUEL TYPE BREAKDOWN:")
    fuel_analysis = results_df.groupby('Primary_Fuel_Type').agg({
        'Total_NOx_Mass': ['count', 'sum', 'mean'],
        'Total_NOx_Mass_Percentage': 'sum'
    }).round(2)
    
    fuel_analysis.columns = ['Facility_Count', 'Total_NOx', 'Avg_NOx_per_Facility', 'Total_Percentage']
    fuel_analysis = fuel_analysis.sort_values('Total_NOx', ascending=False)
    print(fuel_analysis.to_string())
    
    # State analysis
    print(f"\nSTATE BREAKDOWN (Top 5):")
    state_analysis = results_df.groupby('State').agg({
        'Total_NOx_Mass': ['count', 'sum'],
        'Total_NOx_Mass_Percentage': 'sum'
    }).round(2)
    
    state_analysis.columns = ['Facility_Count', 'Total_NOx', 'Total_Percentage']
    state_analysis = state_analysis.sort_values('Total_NOx', ascending=False).head(5)
    print(state_analysis.to_string())
    
    # Cumulative analysis
    cumulative_pct_col = 'Cumulative_Total_NOx_Mass_Percentage'
    if cumulative_pct_col in results_df.columns:
        cumulative_50 = results_df[results_df[cumulative_pct_col] <= 50].shape[0]
        cumulative_80 = results_df[results_df[cumulative_pct_col] <= 80].shape[0]
        
        print(f"\nCUMULATIVE CONCENTRATION:")
        print(f"• Top {cumulative_50} facilities account for 50% of NOx emissions")
        print(f"• Top {cumulative_80} facilities account for 80% of NOx emissions")
        print(f"• Total facilities: {len(results_df)}")

def save_facility_results(results_df, base_filename="facility_emissions_by_plant"):
    """Save results to multiple CSV files."""
    
    # Main comprehensive results
    main_file = f"{base_filename}_comprehensive.csv"
    results_df.to_csv(main_file, index=False)
    print(f"✓ Comprehensive facility results saved to: {main_file}")
    
    # Top 20 facilities summary with coordinates
    top_columns = ['NOx_Rank', 'Facility_Name', 'State', 'Facility_ID', 'Latitude', 'Longitude',
                  'Total_NOx_Mass', 'Total_CO2_Mass', 'Total_SO2_Mass', 'Primary_Fuel_Type', 
                  'Years_in_Operation', 'Total_NOx_Mass_Percentage', 'Total_CO2_Mass_Percentage']
    
    top_file = f"{base_filename}_top20.csv"
    results_df[top_columns].head(20).to_csv(top_file, index=False)
    print(f"✓ Top 20 facilities saved to: {top_file}")
    
    # Geographic data for mapping
    geo_columns = ['NOx_Rank', 'Facility_Name', 'State', 'Latitude', 'Longitude', 
                  'Total_NOx_Mass', 'Total_CO2_Mass', 'Total_SO2_Mass', 'Primary_Fuel_Type']
    
    geo_file = f"{base_filename}_geographic.csv"
    results_df[geo_columns].to_csv(geo_file, index=False)
    print(f"✓ Geographic data saved to: {geo_file}")
    
    # Cumulative statistics
    cumulative_cols = [col for col in results_df.columns if 'Cumulative_' in col]
    cumulative_cols = ['NOx_Rank', 'Facility_Name', 'State', 'Latitude', 'Longitude'] + cumulative_cols
    
    cumulative_file = f"{base_filename}_cumulative_stats.csv"
    results_df[cumulative_cols].to_csv(cumulative_file, index=False)
    print(f"✓ Cumulative statistics saved to: {cumulative_file}")

def main():
    """Main execution function."""
    
    # File path - update this to your actual file path
    csv_file_path = "/net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv"
    
    try:
        # Analyze facilities
        results = analyze_facilities_by_total_emissions(csv_file_path)
        
        # Print results and insights
        print_facility_summary(results)
        
        # Save results to multiple files
        save_facility_results(results)
        
        return results
        
    except FileNotFoundError:
        print(f"Error: File not found at {csv_file_path}")
        print("Please update the csv_file_path variable with the correct path to your data file.")
        return None
    except Exception as e:
        print(f"Error processing data: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Run the facility analysis
if __name__ == "__main__":
    results_df = main()

Reading data from: /net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv
Dataset shape: (8250, 22)
Years covered: 2019 - 2024
Number of unique facilities: 1465
Total facility-year records: 8250

POWER PLANT EMISSIONS ANALYSIS - RANKED BY TOTAL NOx EMISSIONS

TOP EMITTING FACILITIES:
----------------------------------------------------------------------------------------------------------------------------------------------------------------
 NOx_Rank                  Facility_Name State  Latitude  Longitude  Total_NOx_Mass  Total_CO2_Mass  Total_SO2_Mass  Total_Heat_Input_mmBtu    Primary_Fuel_Type  Years_in_Operation  First_Year  Last_Year
        1         New Madrid Power Plant    MO   36.5147   -89.5617        84211.61     36059093.44        69898.18            3.438180e+08                 Coal                   6        2019       2024
        2      Thomas Hill Energy Center    MO   39.5531   -92.6392        57796.40     45321072.43        81900

In [2]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from math import radians, cos, sin, asin, sqrt
from joblib import Parallel, delayed
import time
import os
from itertools import groupby
from operator import itemgetter

# --- 1. Haversine Distance Function ---
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees).
    """
    if any(pd.isna(x) for x in [lon1, lat1, lon2, lat2]):
        return np.inf
    
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  # Radius of earth in kilometers
    return c * r

# --- 2. City Data Loading Function ---
def load_simplemaps_cities(filepath):
    """Loads city data from SimpleMaps CSV, cleans, and prepares for use."""
    print(f"Loading city data from: {filepath}")
    if not os.path.exists(filepath):
        print(f"City file not found: {filepath}")
        return None
    
    cities_df = pd.read_csv(filepath, encoding='utf-8', low_memory=False)
    
    # Convert coordinates and population to numeric
    cities_df['latitude'] = pd.to_numeric(cities_df['latitude'], errors='coerce')
    cities_df['longitude'] = pd.to_numeric(cities_df['longitude'], errors='coerce')
    cities_df['population'] = pd.to_numeric(cities_df['population'], errors='coerce')
    
    # Drop cities with invalid coordinates
    cities_df.dropna(subset=['latitude', 'longitude'], inplace=True)
    
    print(f"Processed city data: {len(cities_df):,} cities with valid coordinates.")
    return cities_df

# --- 3. Year-specific Worker Function ---
def calculate_yearly_nearby_stats(
    reference_plant_row,
    year_plants_df,  # Plants operating in the same year
    city_tree,
    cities_filtered_df,
    plant_lat_col, plant_lon_col, emission_col,
    city_lat_col, city_lon_col, city_pop_col,
    plant_radii_km,
    city_radii_km
):
    """
    Calculates nearby plant AND nearby city statistics for a single reference plant
    using only plants operating in the same year.
    """
    results = {}
    earth_radius_km = 6371
    ref_index = reference_plant_row.name
    
    # Initialize results with NaN
    for radius in plant_radii_km:
        results[f'nearby_plants_count_{radius}km'] = np.nan
        results[f'total_emission_{radius}km'] = np.nan
        results[f'percentage_emission_{radius}km'] = np.nan
    for radius in city_radii_km:
        results[f'nearby_cities_count_{radius}km'] = np.nan
        results[f'nearby_cities_pop_{radius}km'] = np.nan
    
    # Check for valid coordinates
    ref_lat = reference_plant_row[plant_lat_col]
    ref_lon = reference_plant_row[plant_lon_col]
    if pd.isna(ref_lat) or pd.isna(ref_lon):
        return pd.Series(results)
    
    ref_emission = reference_plant_row[emission_col]
    
    # --- Part A: Nearby Plant Calculations (Year-specific) ---
    valid_target_plants = year_plants_df.dropna(subset=[plant_lat_col, plant_lon_col])
    target_plant_coords = list(zip(valid_target_plants[plant_lat_col], valid_target_plants[plant_lon_col]))
    
    plant_distances = pd.Series(
        [haversine(ref_lon, ref_lat, target_lon, target_lat)
         for target_lat, target_lon in target_plant_coords],
        index=valid_target_plants.index
    )
    plant_distances = plant_distances.reindex(year_plants_df.index, fill_value=np.inf)
    
    for radius in plant_radii_km:
        within_radius_mask = plant_distances <= radius
        nearby_plants_df = year_plants_df[within_radius_mask]
        
        # Exclude the reference plant itself
        count_nearby = nearby_plants_df[nearby_plants_df.index != ref_index].shape[0]
        total_emissions_nearby = pd.to_numeric(nearby_plants_df[emission_col], errors='coerce').sum()
        
        # Calculate percentage
        if pd.isna(ref_emission):
            percentage = np.nan
        elif total_emissions_nearby > 0:
            percentage = (ref_emission / total_emissions_nearby) * 100
        elif total_emissions_nearby == 0 and ref_emission == 0:
            percentage = 100.0
        elif total_emissions_nearby == 0 and ref_emission > 0:
            percentage = np.inf
        else:
            percentage = np.nan
        
        results[f'nearby_plants_count_{radius}km'] = count_nearby
        results[f'total_emission_{radius}km'] = total_emissions_nearby
        results[f'percentage_emission_{radius}km'] = percentage
    
    # --- Part B: Nearby City Calculations ---
    if city_tree is not None and cities_filtered_df is not None and not cities_filtered_df.empty:
        plant_lat_rad = radians(ref_lat)
        plant_lon_rad = radians(ref_lon)
        max_city_radius_rad = max(city_radii_km) / earth_radius_km
        
        try:
            potential_indices = city_tree.query_ball_point([plant_lat_rad, plant_lon_rad], r=max_city_radius_rad)
            
            if len(potential_indices) > 0:
                nearby_potential_cities = cities_filtered_df.iloc[potential_indices]
                
                city_distances = [
                    haversine(ref_lon, ref_lat, city_lon, city_lat)
                    for city_lon, city_lat in zip(nearby_potential_cities[city_lon_col], nearby_potential_cities[city_lat_col])
                ]
                nearby_potential_cities_with_dist = nearby_potential_cities.copy()
                nearby_potential_cities_with_dist['distance_km'] = city_distances
                
                for r_km in city_radii_km:
                    cities_within_radius = nearby_potential_cities_with_dist[
                        nearby_potential_cities_with_dist['distance_km'] <= r_km
                    ]
                    count = len(cities_within_radius)
                    total_pop = pd.to_numeric(cities_within_radius[city_pop_col], errors='coerce').fillna(0).sum()
                    results[f'nearby_cities_count_{r_km}km'] = count
                    results[f'nearby_cities_pop_{r_km}km'] = int(total_pop)
            else:
                for r_km in city_radii_km:
                    results[f'nearby_cities_count_{r_km}km'] = 0
                    results[f'nearby_cities_pop_{r_km}km'] = 0
        
        except Exception as e:
            print(f"Error during city calculation for plant index {ref_index}: {e}")
    
    return pd.Series(results)

# --- 4. Main Script Logic ---

# Configuration
power_plant_file = '/net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv'
city_file = '/net/fs06/d3/rzhuang/TROPOMI_US/data/worldcities.csv'

# Column names - will be auto-detected from actual data
plant_lat_col = 'Latitude'  # Will be updated after loading data
plant_lon_col = 'Longitude'  # Will be updated after loading data
emission_col = 'CO2 Mass (short tons)'  # Will be updated after loading data
year_col = 'Year'  # Will be updated after loading data
facility_id_col = 'Facility ID'  # Will try to find this too

city_lat_col = 'latitude'
city_lon_col = 'longitude'
city_pop_col = 'population'

# Radii
plant_radii_km = [20, 50, 100]
city_radii_km = [20, 50, 100]
min_city_population = 50000

# Parallelism
n_jobs = 48

# --- Load Power Plant Data ---
print(f"Loading power plants from: {power_plant_file}")
try:
    locations_df = pd.read_csv(power_plant_file)
    print(f"Successfully loaded {len(locations_df)} rows")
    print(f"Available columns: {locations_df.columns.tolist()}")
except FileNotFoundError:
    print(f"Error: Power plant file not found: {power_plant_file}")
    exit()

# --- Auto-detect Column Names ---
# Check if latitude/longitude columns exist with different names
lat_candidates = ['Latitude', 'latitude', 'lat', 'Lat', 'LATITUDE']
lon_candidates = ['Longitude', 'longitude', 'lon', 'Lon', 'LONGITUDE']
year_candidates = ['Year', 'year', 'YEAR', 'yr']
emission_candidates = ['CO2 Mass (short tons)', 'CO2_Mass', 'CO2', 'emissions', 'Emissions']
facility_id_candidates = ['Facility ID', 'facility_id', 'FacilityID', 'ID', 'id']

# Find actual column names
actual_lat_col = None
actual_lon_col = None
actual_year_col = None
actual_emission_col = None
actual_facility_id_col = None

for candidate in lat_candidates:
    if candidate in locations_df.columns:
        actual_lat_col = candidate
        break

for candidate in lon_candidates:
    if candidate in locations_df.columns:
        actual_lon_col = candidate
        break

for candidate in year_candidates:
    if candidate in locations_df.columns:
        actual_year_col = candidate
        break

for candidate in emission_candidates:
    if candidate in locations_df.columns:
        actual_emission_col = candidate
        break

for candidate in facility_id_candidates:
    if candidate in locations_df.columns:
        actual_facility_id_col = candidate
        break

# Update column names based on what was found
if actual_lat_col:
    plant_lat_col = actual_lat_col
    print(f"Using latitude column: {plant_lat_col}")
else:
    print(f"Error: No latitude column found. Candidates were: {lat_candidates}")
    exit()

if actual_lon_col:
    plant_lon_col = actual_lon_col
    print(f"Using longitude column: {plant_lon_col}")
else:
    print(f"Error: No longitude column found. Candidates were: {lon_candidates}")
    exit()

if actual_year_col:
    year_col = actual_year_col
    print(f"Using year column: {year_col}")
else:
    print(f"Error: No year column found. Candidates were: {year_candidates}")
    exit()

if actual_emission_col:
    emission_col = actual_emission_col
    print(f"Using emission column: {emission_col}")
else:
    print(f"Error: No emission column found. Candidates were: {emission_candidates}")
    print("Available columns containing 'CO2' or 'emission':")
    for col in locations_df.columns:
        if 'CO2' in col or 'emission' in col.lower():
            print(f"  - {col}")
    exit()

if actual_facility_id_col:
    facility_id_col = actual_facility_id_col
    print(f"Using facility ID column: {facility_id_col}")
else:
    print(f"Warning: No facility ID column found. Will use index instead.")
    facility_id_col = None

# --- Validate Power Plant Data ---
required_plant_cols = [plant_lat_col, plant_lon_col, emission_col, year_col]
missing_plant_cols = [col for col in required_plant_cols if col not in locations_df.columns]
if missing_plant_cols:
    print(f"Error: Missing required power plant columns: {missing_plant_cols}")
    print(f"Available columns: {locations_df.columns.tolist()}")
    exit()

# Show sample data
print(f"\nSample of loaded data:")
print(locations_df.head())
print(f"\nData types:")
print(locations_df[required_plant_cols].dtypes)

# Convert and clean power plant data
locations_df[plant_lat_col] = pd.to_numeric(locations_df[plant_lat_col], errors='coerce')
locations_df[plant_lon_col] = pd.to_numeric(locations_df[plant_lon_col], errors='coerce')
locations_df[emission_col] = pd.to_numeric(locations_df[emission_col], errors='coerce')
locations_df[year_col] = pd.to_numeric(locations_df[year_col], errors='coerce')

# Drop rows with missing coordinates or year
initial_rows = len(locations_df)
locations_df = locations_df.dropna(subset=[plant_lat_col, plant_lon_col, year_col])
rows_dropped = initial_rows - len(locations_df)
if rows_dropped > 0:
    print(f"Dropped {rows_dropped} power plants due to missing coordinates or year.")

locations_df = locations_df.reset_index(drop=True)

if locations_df.empty:
    print("Error: Power plant DataFrame is empty after cleaning.")
    exit()

print(f"Processing {len(locations_df)} power plant-year records with valid coordinates.")
print(f"Years covered: {sorted(locations_df[year_col].unique())}")

# --- Load and Prepare City Data ---
print("\n--- Loading and Preparing City Data ---")
city_dataframe_sm = load_simplemaps_cities(city_file)
cities_filtered_df = None
city_tree = None

if city_dataframe_sm is not None:
    if city_pop_col in city_dataframe_sm.columns:
        print(f"Filtering cities by population >= {min_city_population:,}")
        
        cities_filtered_df = city_dataframe_sm[
            (city_dataframe_sm[city_pop_col].fillna(-1) >= min_city_population) &
            city_dataframe_sm[city_lat_col].notna() &
            city_dataframe_sm[city_lon_col].notna()
        ].copy()
        
        print(f"Filtered cities: Found {len(cities_filtered_df)} cities meeting criteria.")
        
        if not cities_filtered_df.empty:
            # Build k-d tree
            cities_filtered_df['lat_rad'] = np.radians(cities_filtered_df[city_lat_col])
            cities_filtered_df['lon_rad'] = np.radians(cities_filtered_df[city_lon_col])
            city_coords_rad = cities_filtered_df[['lat_rad', 'lon_rad']].values
            
            try:
                city_tree = cKDTree(city_coords_rad)
                print("Built k-d tree for filtered cities successfully.")
            except Exception as e:
                print(f"Error building k-d tree: {e}. City analysis will be skipped.")
                city_tree = None
                cities_filtered_df = None

# --- Process by Year ---
print(f"\n--- Processing by Year ---")
print(f"Using {n_jobs} cores for parallel processing.")

all_results = []
years = sorted(locations_df[year_col].unique())

for year in years:
    print(f"\nProcessing year {year}...")
    
    # Get all plants for this year
    year_plants_df = locations_df[locations_df[year_col] == year].copy().reset_index(drop=True)
    print(f"  Found {len(year_plants_df)} power plants operating in {year}")
    
    if len(year_plants_df) == 0:
        continue
    
    # Prepare parallel tasks for this year
    tasks = [
        delayed(calculate_yearly_nearby_stats)(
            year_plants_df.loc[idx],
            year_plants_df,  # Only plants from this year
            city_tree,
            cities_filtered_df,
            plant_lat_col, plant_lon_col, emission_col,
            city_lat_col, city_lon_col, city_pop_col,
            plant_radii_km,
            city_radii_km
        ) for idx in year_plants_df.index
    ]
    
    # Execute in parallel for this year
    start_time = time.time()
    print(f"  Starting parallel calculation for {year}...")
    
    year_results_list = Parallel(n_jobs=n_jobs, verbose=1, backend="loky")(tasks)
    
    end_time = time.time()
    print(f"  Year {year} calculation finished in {end_time - start_time:.2f} seconds.")
    
    # Combine results for this year
    if year_results_list:
        year_stats_df = pd.DataFrame(year_results_list, index=year_plants_df.index)
        year_combined_df = pd.concat([year_plants_df, year_stats_df], axis=1)
        all_results.append(year_combined_df)

# --- Combine All Years ---
if all_results:
    print(f"\nCombining results from all {len(all_results)} years...")
    final_results_df = pd.concat(all_results, ignore_index=True)
    
    print("Calculations complete.")
    print(f"Final dataset shape: {final_results_df.shape}")
    
    # Display sample results
    print("\n--- Sample Results ---")
    plant_stat_cols = [f'{stat}_{r}km' for r in plant_radii_km for stat in ['nearby_plants_count', 'total_emission', 'percentage_emission']]
    city_stat_cols = [f'{stat}_{r}km' for r in city_radii_km for stat in ['nearby_cities_count', 'nearby_cities_pop']]
    
    display_cols = [year_col, plant_lat_col, plant_lon_col, emission_col]
    if facility_id_col and facility_id_col in final_results_df.columns:
        display_cols.insert(1, facility_id_col)  # Insert after year_col
    
    display_cols.extend(plant_stat_cols + city_stat_cols)
    display_cols = [col for col in display_cols if col in final_results_df.columns]
    
    if display_cols:
        print(final_results_df[display_cols].head(10))
        
        print(f"\n--- Summary Statistics by Year ---")
        yearly_summary = final_results_df.groupby(year_col)[plant_stat_cols + city_stat_cols].mean()
        print(yearly_summary)
    
    # Save results
    output_filename = '../data/power_plants_with_yearly_nearby_stats.csv'
    try:
        final_results_df.to_csv(output_filename, index=False)
        print(f"\nResults saved to {output_filename}")
    except Exception as e:
        print(f"\nError saving results: {e}")
        
else:
    print("No results generated.")

Loading power plants from: /net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv
Successfully loaded 8250 rows
Available columns: ['Facility ID', 'Year', 'Operating Time Count', 'Sum of the Operating Time', 'Gross Load (MWh)', 'Steam Load (1000 lb)', 'SO2 Mass (short tons)', 'CO2 Mass (short tons)', 'NOx Mass (short tons)', 'Heat Input (mmBtu)', 'Unit Count', 'Facility Name', 'State', 'Latitude', 'Longitude', 'County', 'EPA Region', 'NERC Region', 'Primary Fuel Type', 'SO2 Rate (lbs/mmBtu)', 'CO2 Rate (short tons/mmBtu)', 'NOx Rate (lbs/mmBtu)']
Using latitude column: Latitude
Using longitude column: Longitude
Using year column: Year
Using emission column: CO2 Mass (short tons)
Using facility ID column: Facility ID

Sample of loaded data:
   Facility ID  Year  Operating Time Count  Sum of the Operating Time  \
0            3  2019                 44642                   44567.00   
1            3  2020                 38901                   38848.75 

[Parallel(n_jobs=48)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=48)]: Done 112 tasks      | elapsed:    0.9s
[Parallel(n_jobs=48)]: Done 612 tasks      | elapsed:    3.1s
[Parallel(n_jobs=48)]: Done 1312 tasks      | elapsed:    6.2s
[Parallel(n_jobs=48)]: Done 1426 out of 1426 | elapsed:    7.0s finished
[Parallel(n_jobs=48)]: Using backend LokyBackend with 48 concurrent workers.


  Year 2019 calculation finished in 7.06 seconds.

Processing year 2020...
  Found 1406 power plants operating in 2020
  Starting parallel calculation for 2020...


[Parallel(n_jobs=48)]: Done 112 tasks      | elapsed:    0.9s
[Parallel(n_jobs=48)]: Done 612 tasks      | elapsed:    3.2s
[Parallel(n_jobs=48)]: Done 1406 out of 1406 | elapsed:    6.9s finished
[Parallel(n_jobs=48)]: Using backend LokyBackend with 48 concurrent workers.


  Year 2020 calculation finished in 6.89 seconds.

Processing year 2021...
  Found 1384 power plants operating in 2021
  Starting parallel calculation for 2021...


...

  Year 2022 calculation finished in 6.54 seconds.

Processing year 2023...
  Found 1344 power plants operating in 2023
  Starting parallel calculation for 2023...


[Parallel(n_jobs=48)]: Done 112 tasks      | elapsed:    0.9s
[Parallel(n_jobs=48)]: Done 612 tasks      | elapsed:    3.2s
[Parallel(n_jobs=48)]: Done 1344 out of 1344 | elapsed:    6.6s finished
[Parallel(n_jobs=48)]: Using backend LokyBackend with 48 concurrent workers.


  Year 2023 calculation finished in 6.57 seconds.

Processing year 2024...
  Found 1326 power plants operating in 2024
  Starting parallel calculation for 2024...


[Parallel(n_jobs=48)]: Done 112 tasks      | elapsed:    0.9s
[Parallel(n_jobs=48)]: Done 612 tasks      | elapsed:    3.1s
[Parallel(n_jobs=48)]: Done 1326 out of 1326 | elapsed:    6.6s finished


  Year 2024 calculation finished in 6.66 seconds.

Combining results from all 6 years...
Calculations complete.
Final dataset shape: (8250, 37)

--- Sample Results ---
   Year  Facility ID  Latitude  Longitude  CO2 Mass (short tons)  \
0  2019            3   31.0069   -88.0103            8290059.850   
1  2019            7   34.0128   -85.9708             176047.675   
2  2019            8   33.6446   -87.2003             598479.723   
3  2019            9   31.7569  -106.3750                  0.000   
4  2019           10   32.6017   -87.7811            1123119.768   
5  2019           26   33.2442   -86.4567            5283997.377   
6  2019           47   34.7439   -87.8486               3152.600   
7  2019           51   32.0306   -93.5692            1266205.718   
8  2019           54   37.8824   -84.1025             257369.073   
9  2019           56   31.4858   -87.9106            1072765.288   

   nearby_plants_count_20km  total_emission_20km  percentage_emission_20km  \
0    

In [20]:
stat = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/power_plants_with_yearly_nearby_stats.csv')

In [9]:
power_plant = '/net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv'
annual_emissions_df = pd.read_csv(power_plant)

In [10]:
annual_emissions_df

,Facility ID,Year,Operating Time Count,Sum of the Operating Time,Gross Load (MWh),Steam Load (1000 lb),SO2 Mass (short tons),CO2 Mass (short tons),NOx Mass (short tons),Heat Input (mmBtu),...,State,Latitude,Longitude,County,EPA Region,NERC Region,Primary Fuel Type,SO2 Rate (lbs/mmBtu),CO2 Rate (short tons/mmBtu),NOx Rate (lbs/mmBtu)
0,3,2019,44642,44567.00,12971220.25,0.00,3494.672,8290059.850,2361.074,1.060760e+08,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.065890,0.078152,0.044517
1,3,2020,38901,38848.75,11671959.25,0.00,1283.482,6846726.200,1656.340,9.189587e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.027933,0.074505,0.036048
2,3,2021,41030,40957.25,12221921.00,0.00,3881.670,7577398.025,2365.775,9.791375e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.079288,0.077388,0.048324
3,3,2022,43760,43632.50,12334645.50,0.00,1349.396,7588387.100,1941.880,9.964551e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.027084,0.076154,0.038976
4,3,2023,37933,37866.00,9909175.25,0.00,280.447,5492459.550,1325.482,7.803848e+07,...,AL,31.0069,-88.0103,Mobile County,4,SERC,Pipeline Natural Gas,0.007187,0.070381,0.033970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8245,880109,2023,3672,3672.00,0.00,0.00,0.000,0.000,11.016,1.220206e+06,...,OH,40.5426,-84.1916,Auglaize County,5,NaN,Pipeline Natural Gas,0.000000,0.000000,0.018056
8246,880109,2024,3672,3672.00,0.00,0.00,0.000,0.000,11.016,1.220206e+06,...,OH,40.5426,-84.1916,Auglaize County,5,NaN,Pipeline Natural Gas,0.000000,0.000000,0.018056
8247,880110,2022,5824,5792.42,0.00,321550.29,0.000,0.000,644.406,1.938144e+06,...,TN,36.5493,-82.6342,Hawkins County,4,NaN,Pipeline Natural Gas,0.000000,0.000000,0.664972
8248,880110,2023,7434,7387.48,0.00,434323.10,0.000,0.000,126.504,6.242826e+05,...,TN,36.5493,-82.6342,Hawkins County,4,NaN,Pipeline Natural Gas,0.000000,0.000000,0.405278


In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# Read the annual emissions data
df = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv')

# Calculate total emissions or operating time to rank facilities
# Group by facility to get total metrics across all years
facility_totals = df.groupby(['Facility ID', 'Facility Name']).agg({
    'Operating Time Count': 'sum',
    'Gross Load (MWh)': 'sum',
    'CO2 Mass (short tons)': 'sum',
    'NOx Mass (short tons)': 'sum'
}).reset_index()

# Sort by total CO2 emissions to get top 500
facility_totals = facility_totals.sort_values('CO2 Mass (short tons)', ascending=False)
top_500_facilities = facility_totals.head(500)['Facility ID'].tolist()

# Filter original data for just top 500 facilities
df_top500 = df[df['Facility ID'].isin(top_500_facilities)]

# Get fuel type by year for each facility
fuel_by_year = df_top500.pivot_table(
    index='Facility ID',
    columns='Year',
    values='Primary Fuel Type',
    aggfunc='first'
)

# Count fuel types in 2024 (or latest available year)
latest_year = df_top500['Year'].max()
fuel_counts_latest = df_top500[df_top500['Year'] == latest_year]['Primary Fuel Type'].value_counts()

print(f"=== TOP 500 POWER PLANTS FUEL TYPE SUMMARY ({latest_year}) ===")
print(fuel_counts_latest)
print(f"\nPipeline Natural Gas plants: {fuel_counts_latest.get('Pipeline Natural Gas', 0)}")

# Track fuel type changes
changes = defaultdict(list)
no_change_count = 0

for facility_id in fuel_by_year.index:
    fuel_history = fuel_by_year.loc[facility_id].dropna()
    unique_fuels = fuel_history.unique()
    
    if len(unique_fuels) > 1:
        # This facility changed fuel type
        change_str = " → ".join(fuel_history.astype(str))
        changes[change_str].append(facility_id)
    else:
        no_change_count += 1

print(f"\n=== FUEL TYPE CHANGES (2019-2024) ===")
print(f"Facilities with no change: {no_change_count}")
print(f"Facilities with changes: {len(fuel_by_year) - no_change_count}")

if changes:
    print("\nDetailed changes:")
    for change_pattern, facilities in sorted(changes.items(), key=lambda x: len(x[1]), reverse=True):
        facility_names = df_top500[df_top500['Facility ID'].isin(facilities)]['Facility Name'].unique()
        print(f"\n{change_pattern}: {len(facilities)} facilities")
        if len(facilities) <= 5:
            for name in facility_names[:5]:
                print(f"  - {name}")

# Summary statistics by fuel type over time
print("\n=== FUEL TYPE COUNTS BY YEAR ===")
yearly_fuel_counts = df_top500.groupby(['Year', 'Primary Fuel Type']).size().unstack(fill_value=0)
print(yearly_fuel_counts)

# Additional analysis: Track major fuel transitions
print("\n=== MAJOR FUEL TYPE TRANSITIONS ===")
coal_to_gas = 0
coal_to_other = 0
gas_to_other = 0

for facility_id in fuel_by_year.index:
    fuel_history = fuel_by_year.loc[facility_id].dropna()
    if len(fuel_history) >= 2:
        first_fuel = fuel_history.iloc[0]
        last_fuel = fuel_history.iloc[-1]
        
        if first_fuel == 'Coal' and last_fuel == 'Pipeline Natural Gas':
            coal_to_gas += 1
        elif first_fuel == 'Coal' and last_fuel != 'Coal':
            coal_to_other += 1
        elif first_fuel == 'Pipeline Natural Gas' and last_fuel != 'Pipeline Natural Gas':
            gas_to_other += 1

print(f"Coal to Natural Gas transitions: {coal_to_gas}")
print(f"Coal to other fuel transitions: {coal_to_other}")
print(f"Natural Gas to other fuel transitions: {gas_to_other}")
yearly_fuel_counts.to_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/fuel_type_counts_by_year.csv')

=== TOP 500 POWER PLANTS FUEL TYPE SUMMARY (2024) ===
Primary Fuel Type
Pipeline Natural Gas    305
Coal                    158
Natural Gas              14
Coal Refuse               4
Diesel Oil                2
Other Gas                 2
Process Gas               1
Petroleum Coke            1
Coal, Natural Gas         1
Unknown                   1
Wood                      1
Name: count, dtype: int64

Pipeline Natural Gas plants: 305

=== FUEL TYPE CHANGES (2019-2024) ===
Facilities with no change: 478
Facilities with changes: 22

Detailed changes:

Natural Gas → Natural Gas, Pipeline Natural Gas → Pipeline Natural Gas → Pipeline Natural Gas → Pipeline Natural Gas → Pipeline Natural Gas: 2 facilities
  - Lansing Smith Generating Plant
  - Stanton A

Coal → Coal → Coal → Coal → Diesel Oil → Diesel Oil: 2 facilities
  - Waukegan
  - Morgantown

Coal → Coal → Coal → Coal → Coal → Pipeline Natural Gas: 2 facilities
  - Dan E Karn
  - A B Brown Generating Station

Coal → Pipeline Natural 